# 04 - 肿瘤内科 LoRA 微调（Qwen3-4B）⭐ 云端

**与 03 同结构**，只换数据集和角色 prompt。直接顺序运行即可。

In [1]:
# Step 0: 环境
import os, pathlib, builtins
os.environ.setdefault('HF_ENDPOINT', 'https://hf-mirror.com')
os.environ['WANDB_DISABLED'] = 'true'
_orig = pathlib.Path.read_text
pathlib.Path.read_text = lambda self, encoding=None, errors=None, **kw: _orig(self, encoding=encoding or 'utf-8', errors=errors, **kw)
_orig_open = builtins.open
def _utf8_open(file, mode='r', buffering=-1, encoding=None, errors=None, **kw):
    if 'b' not in mode and encoding is None: encoding = 'utf-8'
    return _orig_open(file, mode, buffering, encoding, errors, **kw)
builtins.open = _utf8_open
print('OK')

OK


In [2]:
import os
# 关键：解决显存碎片OOM，不改动任何实验参数
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


import torch, gc, json, random, inspect
from pathlib import Path
from datasets import Dataset

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

MODEL_NAME = '/root/autodl-tmp/Qwen3-4B-Instruct-2507'  # AutoDL 5090 本地路径
NOTEBOOK_DIR = Path.cwd()
EVAL_DIR = NOTEBOOK_DIR.parent
BACKEND_DIR = EVAL_DIR.parent
DATA_PATH = EVAL_DIR / 'datasets' / 'data' / 'training' / 'oncologist_train.jsonl'
OUTPUT_DIR = EVAL_DIR / 'models' / 'oncologist_qwen3_lora'
ROLE_PROMPT_PATH = BACKEND_DIR / 'workspace' / 'roles' / 'MEDICAL_ONCOLOGIST.md'

MAX_LENGTH = 2048
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 5e-5
NUM_EPOCHS = 5
LORA_R = 32
LORA_ALPHA = 64
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'数据存在: {DATA_PATH.exists()}')

CUDA: True, GPU: NVIDIA GeForce RTX 5090
数据存在: True


In [3]:
# 数据
random.seed(42)
records = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            r = json.loads(line)
            msgs = r.get('messages', [])
            if len(msgs) >= 3 and msgs[-1].get('role') == 'assistant':
                records.append({'messages': msgs})

random.shuffle(records)
n_val = max(40, int(len(records) * 0.05))
train_dataset = Dataset.from_list(records[n_val:])
val_dataset = Dataset.from_list(records[:n_val])
print(f'训练: {len(train_dataset)}, 验证: {len(val_dataset)}')

训练: 4027, 验证: 211


In [4]:
# 模型 + LoRA
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map='auto', trust_remote_code=True,
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

trainable params: 66,060,288 || all params: 4,088,528,384 || trainable%: 1.6157


In [5]:
# 训练
from trl import SFTTrainer, SFTConfig

sft_init_params = set(inspect.signature(SFTConfig.__init__).parameters.keys())
config_kwargs = dict(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=100,
    save_strategy='steps',
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=10,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported() and torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to='none', seed=42,
)
if 'max_length' in sft_init_params:
    config_kwargs['max_length'] = MAX_LENGTH
elif 'max_seq_length' in sft_init_params:
    config_kwargs['max_seq_length'] = MAX_LENGTH
if 'packing' in sft_init_params:
    config_kwargs['packing'] = False

sft_config = SFTConfig(**config_kwargs)

trainer_init_params = set(inspect.signature(SFTTrainer.__init__).parameters.keys())
trainer_kwargs = dict(model=model, args=sft_config, train_dataset=train_dataset, eval_dataset=val_dataset)
if 'processing_class' in trainer_init_params:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_init_params:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)
print('开始训练 Qwen3-4B 肿瘤内科 LoRA...')
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/4027 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/211 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


开始训练 Qwen3-4B 肿瘤内科 LoRA...


Step,Training Loss,Validation Loss
50,0.590140,0.473966
100,0.401900,0.372822
150,0.348962,0.354485
200,0.331620,0.343516
250,0.324511,0.336655
300,0.331588,0.330896
350,0.305119,0.326359
400,0.291377,0.321698
450,0.289850,0.318257
500,0.306600,0.314871


TrainOutput(global_step=1260, training_loss=0.3265874569378202, metrics={'train_runtime': 9256.5393, 'train_samples_per_second': 2.175, 'train_steps_per_second': 0.136, 'total_flos': 6.28383722130561e+17, 'train_loss': 0.3265874569378202})

In [6]:
# 保存
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
with open(OUTPUT_DIR / 'training_info.json', 'w', encoding='utf-8') as f:
    json.dump({
        'base_model': MODEL_NAME, 'role': 'medical_oncologist',
        'train_size': len(train_dataset), 'epochs': NUM_EPOCHS,
    }, f, ensure_ascii=False, indent=2)
print(f'保存到: {OUTPUT_DIR.resolve()}')

保存到: /root/autodl-tmp/ClowTeam_NLP/backend/eval/models/oncologist_qwen3_lora


In [7]:
# 推理测试
model.config.use_cache = True
model.eval()

system_prompt = ROLE_PROMPT_PATH.read_text(encoding='utf-8') if ROLE_PROMPT_PATH.exists() else '你是肿瘤内科医生。'

test_qs = [
    '58 岁男性，肺腺癌 T3N2M0，EGFR 阳性。推荐什么靶向方案？剂量周期？',
    '65 岁女性，乳腺癌 HER2 阳性，已行手术。辅助治疗方案？',
    '72 岁男性，胃癌 T2N1M0，肝肾功能正常。新辅助 vs 辅助化疗？',
]

for q in test_qs:
    messages = [{'role': 'system', 'content': system_prompt}, {'role': 'user', 'content': q}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tokenizer.pad_token_id)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n=== 问: {q} ===\n答:\n{response}\n{"-"*80}')


=== 问: 58 岁男性，肺腺癌 T3N2M0，EGFR 阳性。推荐什么靶向方案？剂量周期？ ===
答:
治疗目标：控制肿瘤生长，延长无进展生存期。推荐方案：奥希替尼 80mg 每日一次口服。剂量周期：持续口服，直至疾病进展或不可耐受毒性。不良反应：皮疹、腹泻、间质性肺炎、肝功能异常。与其他治疗协调：若需同步放化疗，可考虑奥希替尼减量至 60mg qd；若为术后辅助，建议奥希替尼 80mg qd 在术后 4-6 周内开始。
--------------------------------------------------------------------------------

=== 问: 65 岁女性，乳腺癌 HER2 阳性，已行手术。辅助治疗方案？ ===
答:
治疗目标：降低复发风险，延长无病生存。推荐方案：T-DM1（ado-trastuzumab emtansine）1.4mg/kg，每 3 周一次，共 14 个周期。剂量周期：每 3 周静脉输注。不良反应：心脏毒性（LVEF 下降）、骨髓抑制、腹泻、恶心。与其他治疗协调：若 LVEF 下降 >15%，暂停并评估；与放疗可同步或序贯，注意心脏保护。
--------------------------------------------------------------------------------

=== 问: 72 岁男性，胃癌 T2N1M0，肝肾功能正常。新辅助 vs 辅助化疗？ ===
答:
治疗目标：降低复发风险，提高生存。推荐方案：氟尿嘧啶类为基础的化疗。剂量周期：卡培他滨 1250 mg/m² 口服 bid，d1-14，q3w，共 6 周期。不良反应：手足综合征、腹泻、恶心。与其他治疗协调：若术后，后续辅助化疗；若术前，术后继续完成。
--------------------------------------------------------------------------------
